# B13 Qwen3-30B-A3B — Fine-tuning Notebook (Kaggle TPU v5e)

**Canonical venv only:** `b13_agent/.venv` (Python 3.12). Activating it loads `KAGGLE_API_TOKEN` from `.venv/kaggle_api_token`.
**Vaultguard (`~/.kaggle/kaggle.json`) is NEVER read, used, or modified by this notebook.**
**HF account:** `sofienehmz` (personal). Base model `Qwen/Qwen3-30B-A3B` rev `ad44e77bcd18fa416d9da3bd8f70d33ebb85d39` (verified). Training export `data/exports/qwen3-30b-a3b/` (23,200 train / 3,272 val, SHA-verified).
Run cells top-to-bottom. Cells 1-3 are safe locally. Cell 4+ needs the Kaggle TPU kernel (internet ON, dataset attached).

In [ ]:
import os, sys, json, pathlib
print('python:', sys.version.split()[0])
print('venv:', os.environ.get('VIRTUAL_ENV', '<not-activated>'))
t = os.environ.get('KAGGLE_API_TOKEN', '')
print('KAGGLE_API_TOKEN:', ('len=%d prefix=%s...' % (len(t), t[:9])) if t else 'MISSING (activate b13_agent/.venv)')
print('vaultguard file left untouched (no read attempted)')
from huggingface_hub import HfApi
p = HfApi().whoami()
print('HF user:', p.get('name'), '| orgs:', [o.get('name') for o in p.get('orgs', [])], '(personal ops only)')

In [ ]:
# Local export verification (safe: hashes + counts only, no sample text)
import hashlib, json, pathlib
root = pathlib.Path('..').resolve() if pathlib.Path('../data').exists() else pathlib.Path('/home/b13/Desktop/b13 agent/b13_agent')
d = root / 'data' / 'exports' / 'qwen3-30b-a3b'
m, c = json.loads((d/'manifest.json').read_text()), json.loads((d/'config.json').read_text())
assert m['model'] == c['model'], 'model/config mismatch'
print('model:', m['model']['repo_id'], m['model']['revision'])
for f, meta in m['files'].items():
    ok = hashlib.sha256((d/f).read_bytes()).hexdigest() == meta['sha256']
    print(f, 'sha_ok=', ok)
print('rows:', m['rows'])
print('NOTE: agent-skills split has known 18/14 aggregate mismatch (kept as own version, in manifest).')

## Kaggle setup (run these in the Kaggle kernel, not locally)
1. Upload export as private dataset `jhonmccarty/b13-qwen3-30b-a3b-training` (keep spelling exactly as in `dataset-metadata.json`).
2. Attach it to kernel `jhonmccarty/b13-qwen3-30b-a3b-lora-tpu` (TPU on, GPU off, internet on).
3. Run the cells below inside that kernel.

In [ ]:
# Kaggle kernel only: install pinned deps, then tokenizer length check vs 2048 window
%pip install --disable-pip-version-check -r requirements.txt
from transformers import AutoTokenizer
import json
cfg = json.load(open('config.json'))
tok = AutoTokenizer.from_pretrained(cfg['model']['repo_id'], revision=cfg['model']['revision'], use_fast=True)
ml = cfg['model']['max_length']
import statistics
def over(path, n=2000):
    Ls = []
    
    with open(path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n: break
            ids = tok.apply_chat_template(json.loads(line)['messages'], tokenize=True, add_generation_prompt=False, truncation=False)
            Ls.append(len(ids))
    Ls.sort()
    return {'n': len(Ls), 'p50': statistics.median(Ls), 'p95': Ls[int(len(Ls)*0.95)], 'max': max(Ls), 'over_%d' % ml: sum(1 for x in Ls if x > ml)}
print('train:', over('/kaggle/input/b13-qwen3-30b-a3b-training/train.jsonl'))
print('val:', over('/kaggle/input/b13-qwen3-30b-a3b-training/validation.jsonl'))
print('BLOCKER if over_2048 > 0: truncate or raise max_length before --run-training.')

In [ ]:
# Kaggle kernel only: preflight then full run (spends TPU quota only here)
!python train_tpu.py --validate-only
!python train_tpu.py
# full LoRA run (after preflight passes):
# !python train_tpu.py --run-training

## After training: publish adapter privately under YOUR personal HF account
`python training/scripts/publish_adapter_to_hf.py /kaggle/working/b13-qwen3-30b-a3b` — refuses non-private repos, never targets orgs (vaultguard safe).
Legacy `training_environment/train_b13*.py` + `kaggle_tpu_train*.py` are DEPRECATED (stale `hdd10-2026` paths, DeepSeek-V4-Pro 1.6T not fittable on GTX 1650 Ti 4GB). Do not use.